### Data Ingestion

#### Langchain Basics

Basics:
LangChain Doc = 1. page_component(str), 2. metadata(dict)

Loaders:
1.PDF, 2.CSV, 3.WebBase, 4.Directory

In [4]:
#### Document Data Structure

from langchain_core.documents import Document

In [5]:
doc = Document(
    page_content="this is the main text content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages": 1,
        "author": "Mulla Modi",
        "data_created":"2025-01-01"
    }
    # Useful for filters
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Mulla Modi', 'data_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [6]:
## Create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [7]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

print('☑️Sample text files created!')

☑️Sample text files created!


In [8]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding='utf-8')
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [9]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

documents = dir_loader.load()
documents

100%|██████████| 2/2 [00:00<00:00, 954.66it/s]


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [10]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdfs",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()
pdf_documents

  0%|          | 0/5 [00:00<?, ?it/s]

MuPDF error: library error: zlib error: incorrect header check



 20%|██        | 1/5 [00:00<00:01,  2.91it/s]

MuPDF error: library error: zlib error: incorrect header check



100%|██████████| 5/5 [00:00<00:00, 12.58it/s]


[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdfs\\Asfaan Resume.pdf', 'file_path': '..\\data\\pdfs\\Asfaan Resume.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Real Resume', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content=''),
 Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'file_path': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Asfaan Resume Latest 2', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Syed Asfaan Hussain \nasfaanisworking@gmail.com | +916302770179 | Linkedin  \nQA Engineer | Product Support Enthusiast | UI & API Testing | Python | SQL | Power BI \n \nExperi

In [11]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### RAG pipelines - Data Ingestion -> Vector DB Pipeline

In [12]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
### Read All the pdfs inside dir
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdfs")

incorrect startxref pointer(1)
parsing for Object Streams
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check


Found 5 PDF files to process

Processing: Asfaan Resume.pdf
  ✓ Loaded 1 pages

Processing: Asfaan_Hussain_CV.pdf


incorrect startxref pointer(3)
parsing for Object Streams
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check
Error -3 while decompressing data: incorrect header check


  ✓ Loaded 1 pages

Processing: Asfaan_Hussain_CV_DS.pdf
  ✓ Loaded 1 pages

Processing: Asfaan_Hussain_CV_ML.pdf
  ✓ Loaded 1 pages

Processing: Asfaan_Hussain_CV_SWE.pdf
  ✓ Loaded 1 pages

Total documents loaded: 5


In [14]:
# Text Splitting into chunks

def split_documents(documents, chunk_size = 60, chunk_overlap = 10):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n', '\n', ' ', '']
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of chunk
    if split_docs:
        print(f"""\nExample Chunk:
            \nContent: {split_docs[0].page_content}....
            \nMetaData: {split_docs[0].metadata}""")
        
    return split_docs

In [15]:
chunks = split_documents(all_pdf_documents)
chunks

Split 5 documents into 244 chunks

Example Chunk:
            
Content: Syed  Asfaan  Hussain  asfaanisworking@gmail.com |....
            
MetaData: {'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Asfaan Resume Latest 2', 'source': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Asfaan_Hussain_CV.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Asfaan Resume Latest 2', 'source': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Asfaan_Hussain_CV.pdf', 'file_type': 'pdf'}, page_content='Syed  Asfaan  Hussain  asfaanisworking@gmail.com |'),
 Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Asfaan Resume Latest 2', 'source': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Asfaan_Hussain_CV.pdf', 'file_type': 'pdf'}, page_content='|  +916302770179  |  Linkedin  QA  Engineer  |  Product'),
 Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Asfaan Resume Latest 2', 'source': '..\\data\\pdfs\\Asfaan_Hussain_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'sour

#### Embeddings

In [16]:
import numpy as np
import uuid
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using sentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded sucessfully")
            print(f"Embedding dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        embeddings = self.model.encode(texts, show_progress_bar = True)
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6315.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded sucessfully
Embedding dimensions: 384


#### Vector Store

In [18]:
class VectorStore:
    def __init__(self, collection_name:str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        "Initialize ChromaDB client and collection"
        try:
            # Creation of Persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector storee initializeeed. Collection : {self.collection_name}")
            print(f"Existting docs in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        # if len(all_pdf_documents) != len(embeddings):
        #     raise ValueError("Number of docs != No. of embeddings")
        
        print(f"Adding {len(all_pdf_documents)} docs to VectorDB")

        # Prep data for store

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(all_pdf_documents, embeddings)):
            # Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # doc content
            documents_text.append(doc.page_content)

            # embeddings
            embeddings_list.append(embedding.tolist())

            # Add to collection
            try:
                self.collection.add(
                    ids = ids,
                    metadatas = metadatas,
                    embeddings = embeddings_list,
                    documents = documents_text
                )
                print(f"Sucessfully added docs to vector store")
                print(f"Total doocs in colleection : {self.collection.count()}")

            except Exception as e:
                print(f"Erroor")
                raise

vectorStore = VectorStore()
vectorStore


Vector storee initializeeed. Collection : pdf_documents
Existting docs in collection: 15


In [19]:
# convert text too embeddings
texts = [doc.page_content for doc in chunks]

# generate embeddings
embeddings = embedding_manager.generate_embeddings(texts=texts)

# store in VecDB
vectorStore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 8/8 [00:00<00:00, 16.40it/s]


Adding 5 docs to VectorDB
Sucessfully added docs to vector store
Total doocs in colleection : 16
Sucessfully added docs to vector store
Total doocs in colleection : 17
Sucessfully added docs to vector store
Total doocs in colleection : 18
Sucessfully added docs to vector store
Total doocs in colleection : 19
Sucessfully added docs to vector store
Total doocs in colleection : 20


### RAG Retriever

In [20]:
class RAGRetriever:
    def __init__(self, vectorStore: VectorStore, embedding_manager: EmbeddingManager):
        self.vectorStore = vectorStore
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5,  score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving docs for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vectorStore.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Find similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} docs after filtering")
                
            else:
                print("no docs found")
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval {e}")
            return []


rag_retriever = RAGRetriever(vectorStore, embedding_manager)

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("Asfaan Hussain")

Retrieving docs for query: 'Asfaan Hussain'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.37it/s]

Retrieved 2 docs after filtering


[{'id': 'doc_14364577_0',
  'content': 'Syed  Asfaan  Hussain  asfaanisworking@gmail.com |  +916302770179  |  Linkedin  Associate  Software  Engineer  |  QA  &  Automation  |  C#,  .NET  Core  |  JavaScript  |  SQL  |  AWS  |  Angular  \n \nExperience  Apprentice  -  Quality  Engineering  &  IT  Project  Management,  S&P  Global  –  Hyderabad,  IN   Sep  2024  –  Sep  2025  ●  Designed  and  executed  automated  test  scripts  using  Selenium  WebDriver  and  integrated  them  into  TeamCity .  ●  Debugged  APIs  to  ensure  proper  functionality,  smooth  integration,  and  enhanced  performance.  ●  Contributed  to  managing  a  large  test  repository  using  Git ,  ensuring  framework  consistency  across  teams.  ●  Collaborated  with  developers  and  analysts  to  create  automated  test  cases  and  drive  a  25%  defect  reduction .  ●  Assisted  project  managers  in  planning,  tracking,  and  delivering  multiple  cross-functional  initiatives  on  time  and  within  scope.

### Pipeline to integrate VectorDB context with LLM output

In [26]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key = groq_api_key,
    model_name = "openai/gpt-oss-20b",
    temperature = 0.1,
    max_tokens = 1024
)

def rag_simple(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context fooound too answer thee question"
    
    prompt = f""" Use the following context to answer the question concisely
    Context: 
    {context}
    
    Question: 
    {query}
    
    Answer:"""

    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [30]:
answer1 = rag_simple("What does Asfaan Hussain do?", rag_retriever, llm)
print(answer1)

Retrieving docs for query: 'What does Asfaan Hussain do?'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 104.68it/s]

Retrieved 2 docs after filtering


Asfaan Hussain is an Associate Software Engineer specializing in QA and automation. He develops and maintains automated test scripts (Selenium, WebDriverIO, .NET Core), manages test repositories, reduces defects, and supports project delivery and dashboards at S&P Global. He also has experience as an operations analyst intern and a data‑entry specialist, working with Python, C#, JavaScript, SQL, AWS, and Angular.


In [31]:
answer2 = rag_simple("What skills does Asfaan Hussain have?", rag_retriever, llm)
print(answer2)

Retrieving docs for query: 'What skills does Asfaan Hussain have?'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 104.50it/s]

Retrieved 2 docs after filtering


**Asfaan Hussain’s skill set**

- **Languages**: Python, C#, JavaScript (TypeScript), SQL  
- **Frameworks / Libraries**: Angular, Mocha, NX, WebdriverIO, Selenium, .NET  
- **Testing**: End‑to‑End, UI, API, Data testing  
- **CI/CD & Version Control**: TeamCity, Git, GitHub  
- **Project & Issue Tracking**: Jira, Miro  
- **Dashboards & Reporting**: Power BI, Excel, Google Sheets  
- **Other Technical**: AWS, UNIX, Email APIs, SQL databases  
- **Soft Skills**: Problem‑solving, research, teamwork, adaptability, communication, excellent typing, data entry.


In [33]:
answer3 = rag_simple("What accomplishments does Asfaan Hussain have?", rag_retriever, llm)
print(answer3)

Retrieving docs for query: 'What accomplishments does Asfaan Hussain have?'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 89.46it/s]

Retrieved 2 docs after filtering


**Key Accomplishments of Asfaan Hussain**

- **Automation & QA Impact (S&P Global)**
  - Designed & executed Selenium‑based test scripts, integrated into TeamCity CI/CD.
  - Debugged APIs, improving integration and performance.
  - Managed a large Git‑based test repository, ensuring framework consistency.
  - Created automated test cases that reduced defects by **25 %**.
  - Built Jira dashboards for sprint/milestone visibility.
  - Coordinated disaster‑recovery drills for multiple services.

- **Process & Efficiency Improvements**
  - Revamped video production workflows at Narayana’s Learning App, cutting project completion time by **8 %**.
  - Developed Python + Email‑API automation scripts at Sykes, slashing manual reporting time by **80 %**.
  - Trained and documented onboarding for 15+ analysts, boosting operational throughput.

- **Academic Projects**
  - Built a diabetes‑prediction web app with **94 %** accuracy using machine learning.
  - Developed a retinal‑disease classificat

In [39]:
def rag_advanced(query, retriever, llm, top_k = 3, min_score = 0.2, return_context = False):
    """
        Advanced RAG function that returns the answer, sources, confidence and context.
    """
    results = retriever.retrieve(query, top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [],  'confidence': 0.0, 'context': ''}

    context = "\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '......'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""
        Use the following context too answer the question concisely.\n
        Context: {context}
        Question: {query}
        Answer:
    """
    response = llm.invoke([prompt.format(context = context, query = query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output
    

In [40]:
result = rag_advanced("What skills can asfaan hussain work on", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print('Answers: ', result['answer'])
print('Sources: ', result['sources'])
print('Confidence: ', result['confidence'])
print('Context: ', result['context'])

Retrieving docs for query: 'What skills can asfaan hussain work on'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.24it/s]

Retrieved 2 docs after filtering


Answers:  **Skills Asfaan Hussain could develop further**

| Category | Suggested Focus |
|----------|-----------------|
| **Cloud & DevOps** | Advanced AWS services (Lambda, ECS/EKS, CloudFormation), Docker/Kubernetes, Terraform, GitOps, automated release pipelines |
| **Testing & Quality** | Performance & load testing (JMeter, k6), security testing (OWASP ZAP, SAST/DAST), test‑driven development, BDD frameworks (Cucumber) |
| **Data & Analytics** | Big‑data processing (Spark, Hadoop), advanced SQL & NoSQL (MongoDB, DynamoDB), data‑visualization (Tableau, Power BI) |
| **Machine Learning** | Model deployment (MLflow, SageMaker), model monitoring, explainable AI, reinforcement learning basics |
| **Programming** | Deepen Python (asyncio, multiprocessing), TypeScript best practices, .NET Core advanced patterns |
| **Soft Skills** | Agile facilitation, stakeholder communication, mentorship, technical writing, project budgeting |

These areas complement his current expertise in QA, automa

In [46]:
result = rag_advanced("What are fake skills of Asfaan Hussain?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print('Answers: ', result['answer'])
print('Sources: ', result['sources'])
print('Confidence: ', result['confidence'])
print('Context: ', result['context'])

Retrieving docs for query: 'What are fake skills of Asfaan Hussain?'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.77it/s]

Retrieved 2 docs after filtering


Answers:  **Fake (or unsupported) skills listed for Asfaan Hussain**

- Data Entry  
- Excellent Typing  
- UNIX  
- Miro  
- Power BI  
- Mocha  
- NX  

These items are either generic soft‑skills or tools/technologies that aren’t evidenced in the experience section of the résumé.
Sources:  [{'source': 'Asfaan Resume.pdf', 'page': 0, 'score': 0.10593855381011963, 'preview': 'Syed  Asfaan  Hussain  asfaanisworking@gmail.com |  +916302770179  |  Linkedin  Associate  Software  Engineer  |  QA  &  Automation  |  C#,  .NET  Core  |  JavaScript  |  SQL  |  AWS  |  Angular  \n \nExperience  Apprentice  -  Quality  Engineering  &  IT  Project  Management,  S&P  Global  –  Hyderab......'}, {'source': 'Asfaan Resume.pdf', 'page': 0, 'score': 0.10593855381011963, 'preview': '......'}]
Confidence:  0.10593855381011963
Context:  Syed  Asfaan  Hussain  asfaanisworking@gmail.com |  +916302770179  |  Linkedin  Associate  Software  Engineer  |  QA  &  Automation  |  C#,  .NET  Core  |  JavaScript  |  